In [5]:
!pip install -q mne moabb tensorflow

import numpy as np
import matplotlib.pyplot as plt
import mne
from moabb.datasets import BNCI2014001
from moabb.paradigms import MotorImagery


from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Flatten, Dense, Activation, Dropout,
                                     Conv2D, MaxPooling2D, AveragePooling2D,
                                     SeparableConv2D, DepthwiseConv2D,
                                     BatchNormalization, SpatialDropout2D)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("imported success.")
print("Ready to build EEGNet.")

imported success.
Ready to build EEGNet.


In [9]:
from moabb.datasets import BNCI2014001
from moabb.paradigms import MotorImagery

dataset = BNCI2014001()
subject_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9]


paradigm = MotorImagery(n_classes=4, fmin=8, fmax=32, tmin=0.5, tmax=3.5)

print("Augmentation...")
X, y, metadata = paradigm.get_data(dataset=dataset, subjects=subject_ids)


le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_dl = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)

X_train, X_test, y_train, y_test = train_test_split(X_dl, y_encoded, test_size=0.2, random_state=42)



print("="*30)
print(f"Data Power Up!")
print(f" Original Trials: {len(y) // len(subject_ids)} per person")
print(f" Total Training Data: {X_train.shape[0]} samples")
print("="*30)


Augmentation...
Data Power Up!
 Original Trials: 576 per person
 Total Training Data: 4147 samples


In [10]:
def EEGNet(nb_classes, Chans=22, Samples=1001,
           dropoutRate=0.5, kernLength=64, F1=8,
           D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):

    input1   = Input(shape=(Chans, Samples, 1))

    block1       = Conv2D(F1, (1, kernLength), padding='same',
                          input_shape=(Chans, Samples, 1),
                          use_bias=False)(input1)
    block1       = BatchNormalization()(block1)

    block1       = DepthwiseConv2D((Chans, 1), use_bias=False,
                                   depth_multiplier=D,
                                   depthwise_constraint=max_norm(1.))(block1)
    block1       = BatchNormalization()(block1)
    block1       = Activation('elu')(block1)
    block1       = AveragePooling2D((1, 4))(block1)
    block1       = Dropout(dropoutRate)(block1)

    block2       = SeparableConv2D(F2, (1, 16),
                                   use_bias=False, padding='same')(block1)
    block2       = BatchNormalization()(block2)
    block2       = Activation('elu')(block2)
    block2       = AveragePooling2D((1, 8))(block2)
    block2       = Dropout(dropoutRate)(block2)

    flatten      = Flatten(name='flatten')(block2)


    dense        = Dense(nb_classes, name='dense',
                         kernel_constraint=max_norm(norm_rate))(flatten)
    softmax      = Activation('softmax', name='softmax')(dense)

    return Model(inputs=input1, outputs=softmax)

sample_length = X_train.shape[2]

print(f"Input Length: {sample_length}")
model = EEGNet(nb_classes=4, Chans=22, Samples=sample_length)


model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
checkpointer = ModelCheckpoint(filepath='/content/best_model.h5', verbose=1, save_best_only=True)


print("Starting Training (Cross-Subject)...")
history = model.fit(X_train, y_train, batch_size=64, epochs=50,
                    validation_data=(X_test, y_test),
                    callbacks=[checkpointer], verbose=1)


model.load_weights('/content/best_model.h5')

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"Final Accuracy (9 Subjects): {accuracy * 100:.2f}%")
print("Model Saved as: '/content/best_model.h5'")
print("="*40)

Input Length: 751


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting Training (Cross-Subject)...
Epoch 1/50


 50%|██████████████████▎                  | 21.7M/43.8M [23:38<24:00, 15.3kB/s]


65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2597 - loss: 1.3931
Epoch 1: val_loss improved from inf to 1.36865, saving model to /content/best_model.h5


...

65/65 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.5255 - loss: 1.0692 - val_accuracy: 0.5776 - val_loss: 1.0351
Epoch 37/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5335 - loss: 1.0738
Epoch 37: val_loss improved from 1.03506 to 1.02755, saving model to /content/best_model.h5


65/65 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.5334 - loss: 1.0739 - val_accuracy: 0.5892 - val_loss: 1.0275
Epoch 38/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5088 - loss: 1.0827
Epoch 38: val_loss did not improve from 1.02755
65/65 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.5089 - loss: 1.0827 - val_accuracy: 0.5757 - val_loss: 1.0375
Epoch 39/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5276 - loss: 1.0762
Epoch 39: val_loss did not improve from 1.02755
65/65 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.5276 - loss: 1.0763 - val_accuracy: 0.5468 - val_loss: 1.0409
Epoch 40/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5434 - loss: 1.0679
Epoch 40: val_loss did not improve from 1.02755
65/65 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.5433 - loss: 1.0681 - val_accuracy: 0.5873 - val_loss: 1.0303
Epoch 41/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5318 - loss: 1.0661
Epoch 41: val_loss did not improve from 1.02755
65/65 ━━━━━━

65/65 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.5191 - loss: 1.0778 - val_accuracy: 0.5834 - val_loss: 1.0136
Epoch 43/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5440 - loss: 1.0528
Epoch 43: val_loss did not improve from 1.01359
65/65 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.5438 - loss: 1.0530 - val_accuracy: 0.5921 - val_loss: 1.0189
Epoch 44/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5241 - loss: 1.0725
Epoch 44: val_loss did not improve from 1.01359
65/65 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.5241 - loss: 1.0725 - val_accuracy: 0.5796 - val_loss: 1.0245
Epoch 45/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5414 - loss: 1.0530
Epoch 45: val_loss did not improve from 1.01359
65/65 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.5411 - loss: 1.0532 - val_accuracy: 0.5651 - val_loss: 1.0397
Epoch 46/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5233 - loss: 1.0593
Epoch 46: val_loss did not improve from 1.01359
65/65 ━━━━━━

65/65 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.5396 - loss: 1.0649 - val_accuracy: 0.5738 - val_loss: 1.0130
Epoch 50/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5319 - loss: 1.0682
Epoch 50: val_loss improved from 1.01299 to 1.01148, saving model to /content/best_model.h5


65/65 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.5318 - loss: 1.0682 - val_accuracy: 0.5805 - val_loss: 1.0115

Final Accuracy (9 Subjects): 58.05%
Model Saved as: '/content/best_model.h5'


In [11]:

def add_gaussian_noise(data, noise_level=0.05):

    noise = np.random.normal(0, noise_level, data.shape)
    noisy_data = data + noise
    return noisy_data


print("Simulating Real-world Noise")


X_train_noisy = add_gaussian_noise(X_train, noise_level=0.1)


model_robust = EEGNet(nb_classes=4, Chans=22, Samples=X_train.shape[2])
model_robust.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Starting Robust Training (Noise Injection Mode)")

history_robust = model_robust.fit(X_train_noisy, y_train,
                                  batch_size=64, epochs=50,
                                  validation_data=(X_test, y_test),
                                  verbose=1)


loss, acc = model_robust.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"Robust Model Accuracy: {acc * 100:.2f}%")
print("="*40)

model_robust.save_weights('/content/best_model_robust.h5')
print("Saved Robust Model as: '/content/best_model_robust.h5'")
PLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLL

Simulating Real-world Noise
Starting Robust Training (Noise Injection Mode)
Epoch 1/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.2591 - loss: 1.3869 - val_accuracy: 0.3134 - val_loss: 1.3725
Epoch 2/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.3149 - loss: 1.3463 - val_accuracy: 0.3173 - val_loss: 1.3615
Epoch 3/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.3570 - loss: 1.3116 - val_accuracy: 0.4214 - val_loss: 1.3125
Epoch 4/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.4019 - loss: 1.2917 - val_accuracy: 0.4581 - val_loss: 1.2634
Epoch 5/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.4271 - loss: 1.2474 - val_accuracy: 0.4889 - val_loss: 1.2236
Epoch 6/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.4184 - loss: 1.2390 - val_accuracy: 0.4619 - val_loss: 1.2052
Epoch 7/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.4392 - loss: 1.2125 - val_accuracy: 0.4764 - val_loss: 1.1803
Epoch 8/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 140

ValueError: The filename must end in `.weights.h5`. Received: filepath=/content/best_model_robust.h5

In [12]:
model_robust.save('/content/best_model_robust.h5')

print("✅ Saved Full Model as: '/content/best_model_robust.h5'")

✅ Saved Full Model as: '/content/best_model_robust.h5'
